In [0]:
import os , sys

current = os.getcwd()
reusable = os.path.join(current , ".." , "..")
print(reusable)
sys.path.append(reusable)
from utils.modules import *


In [0]:
%fs ls "abfss://bronze@spotifyproject.dfs.core.windows.net/DimDate/"

In [0]:
df_date = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimDate/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@spotifyproject.dfs.core.windows.net/DimDate/*")

In [0]:
obj = Transform()
df_date = obj.dropColumn(df_date,['_rescued_data'])

In [0]:


df_date.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimDate/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@spotifyproject.dfs.core.windows.net/DimDate/data")\
            .toTable("spotify_catalog.silver.DimDate")

In [0]:
%sql
select * from spotify_catalog.silver.dimdate